In [33]:
from datasets import Dataset
from retrival_methods import retrival_pipeline 
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import HuggingFaceEmbeddings as RagasHuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from models import build_llms
from ragas.run_config import RunConfig
from ragas.llms import llm_factory
from models import get_embedding_model
from openai import OpenAI
import re

#making the llm_with_fallback
llm_grok, llm_openrouter, lazy_llm=build_llms()

client = OpenAI(api_key="pranshu123", base_url="http://localhost:8005/v1")
evaluator_llm = llm_factory("Qwen/Qwen3-14B-AWQ", client=client)
run_config = RunConfig(timeout=180, max_retries=5,max_wait=60,max_workers=10)



def main_retrival_pipeline(query: str,collection="DemoRAG"):
    
    
    pipeline = retrival_pipeline(collection=collection)
    all_retrieval_results = pipeline.multiquery_RRM(query)

    fused_results = pipeline.reciprocal_rank_fusion(
        all_retrieval_results, k=60, verbose=False
    )
    
    reranked_docs_c = pipeline.reranker_chunks()
    
    response = pipeline.generate_final_answer(chunks=reranked_docs_c, query=query)
    
    return response,reranked_docs_c
questions = [
    "What is Easy Build?",
]

ground_truths = [
    "A B2B2C online channel platform.",
]
ragas_rows = []

for question, ground_truth in zip(questions, ground_truths):
    answer,contexts= main_retrival_pipeline(query=question)
    answer=re.sub(r"<think>.*?</think>","",answer,flags=re.DOTALL,).strip()
    ragas_rows.append({
        "user_input":question,
        "retrieved_contexts":[docs.page_content for docs in contexts],
        "response": answer,
        "reference":ground_truth,
    }
    )


from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_list(ragas_rows)

-----Hybrid Retriever intialised

=====Results for Query 1:What is the definition of Easy Build?===
Retrived 8 documents:

Document1:
Moreover, ordinary Portland Cement is manufactured by pulverizing clinker consisting of hydraulic calcium silicates, usually containing one or more fo
Document2:
4. Digital Platform Roadmap & Future Growth

Over the next five years, EASY BUILD will enhance its AI integrations for price forecasting, automated ro
Document3:
Table of Contents

1. Founding Story & Vision 2. B2B2C Business Model & Platform Expansion 3. Department Breakdown & Governance 4. Digital Platform Ro
Document4:
EASY BUILD: CORPORATE HISTORY AND OVERVIEW

Document ID: EB-CORP-001

Department: Corporate Strategy

Owner: Rajesh Kumar (CEO)

Version: 4.2

Effecti
Document5:
Appendix: Corporate Organization Chart

The following diagram outlines the key reporting structure of EASY BUILD executive leadership.

Figure 1.1: Ex
Document6:
<think>
Okay, let's tackle this query. The user wants a 

In [ ]:
evaluator_embeddings = RagasHuggingFaceEmbeddings(model="BAAI/bge-small-en-v1.5",device="cpu")


from ragas import evaluate
from ragas.metrics.collections import (
    AnswerCorrectness,
    Faithfulness,
    ContextPrecision,
    ContextRecall,
    AnswerRelevancy
)

import json

answer_correctness = AnswerCorrectness(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    strictness=1,
)

faithfulness = Faithfulness(
    llm=evaluator_llm,
)

context_precision = ContextPrecision(
    llm=evaluator_llm,
)

context_recall = ContextRecall(
    llm=evaluator_llm,
)


import traceback

try:
    scores = evaluate(
        evaluation_dataset,
        metrics=[
            answer_correctness,
            answer_relevancy,
            faithfulness,
            context_precision,
            context_recall,
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=run_config,
    )
except Exception:
    traceback.print_exc()

aggregate_scores = dict(scores)

scores_df = scores.to_pandas()
print(scores_df)
per_row_scores = scores_df.to_dict(orient="records")

merged =[]

for row,score_row  in zip(ragas_rows,per_row_scores):
    merged.append({**row,**score_row})

output = {
    "aggregate_scores":aggregate_scores,
    "results":merged,
}

with open("eval_results.json",'w') as f :
    json.dump(output,f,indent=2,ensure_ascii=False )
print(f"Saved {len(merged)} results to eval_resuls.json")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TypeError: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]

In [34]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(ragas_rows)
print(ragas_rows[0])
print(evaluation_dataset.features())

{'user_input': 'What is Easy Build?', 'retrieved_contexts': ['Table of Contents\n\n1. Founding Story & Vision 2. B2B2C Business Model & Platform Expansion 3. Department Breakdown & Governance 4. Digital Platform Roadmap & Future Growth\n\n1. Founding Story & Vision\n\nEASY BUILD was founded in 2021 with the vision of organizing the highly fragmented Indian construction and building-materials supply chain. Historically, homeowners and small contractors struggled with fluctuating prices, inconsistent material quality, and unreliable delivery schedules. Our platform integrates physical experience centers with digital commerce, creating transparency and trust.', '4. Digital Platform Roadmap & Future Growth\n\nOver the next five years, EASY BUILD will enhance its AI integrations for price forecasting, automated route optimization, and multimodal customer support chatbots. These systems run on PostgreSQL and local vector databases to drive localized supply-chain logistics.', 'EASY BUILD: COR

In [35]:
print(ragas_rows is rows)

False


In [20]:
evaluation_dataset.features

<bound method RagasDataset.features of EvaluationDataset(features=['reference'], len=1)>

In [10]:
print(rows)

[{'question': 'What is Easy Build?', 'contexts': ['Table of Contents\n\n1. Founding Story & Vision 2. B2B2C Business Model & Platform Expansion 3. Department Breakdown & Governance 4. Digital Platform Roadmap & Future Growth\n\n1. Founding Story & Vision\n\nEASY BUILD was founded in 2021 with the vision of organizing the highly fragmented Indian construction and building-materials supply chain. Historically, homeowners and small contractors struggled with fluctuating prices, inconsistent material quality, and unreliable delivery schedules. Our platform integrates physical experience centers with digital commerce, creating transparency and trust.', '4. Digital Platform Roadmap & Future Growth\n\nOver the next five years, EASY BUILD will enhance its AI integrations for price forecasting, automated route optimization, and multimodal customer support chatbots. These systems run on PostgreSQL and local vector databases to drive localized supply-chain logistics.', 'EASY BUILD: CORPORATE HIST